In [ ]:
# ── Cell 1: CPU Setup ────────────────────────────────────────────────────────
# No GPU required — uses pre-saved .npy feature arrays

import subprocess, sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fmd'
    IN_COLAB = True
except ImportError:
    PROJECT_DIR = '/Users/aman2394/Desktop/ICWSM-2026-FMD'
    IN_COLAB = False
    print('Running locally — skipping Drive mount')

os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
sys.path.insert(0, f'{PROJECT_DIR}/src')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'lightgbm>=4.3.0', 'scikit-learn>=1.4.0',
    'pandas', 'numpy', 'matplotlib', 'scipy'
], check=True)

print(f'PROJECT_DIR = {PROJECT_DIR}')
print('CPU setup complete.')

In [ ]:
# ── Cell 2: Load All Feature Arrays ──────────────────────────────────────────

import numpy as np
import pandas as pd

t1 = np.load(f'{PROJECT_DIR}/feature_cache/tier1_features.npy')
t2 = np.load(f'{PROJECT_DIR}/feature_cache/tier2_oof_preds.npy')
t3 = np.load(f'{PROJECT_DIR}/feature_cache/tier3_features.npy')
y  = np.load(f'{PROJECT_DIR}/feature_cache/labels.npy')

print('Loaded feature arrays:')
print(f'  Tier 1 : {t1.shape}')
print(f'  Tier 2 : {t2.shape}')
print(f'  Tier 3 : {t3.shape}')
print(f'  Labels : {y.shape}')
print()

for name, arr in [('Tier 1', t1), ('Tier 2', t2), ('Tier 3', t3)]:
    assert not np.isnan(arr).any(), f'NaN values in {name}'
print('No NaN values detected in any feature array.')

# Break out Tier 1 sub-blocks for granular ablation
# NLI: 5, Embedding: t1.shape[1]-12, Coherence: 4, Epistemic: 3
nli_end   = 5
emb_end   = t1.shape[1] - 7   # all but last 7 cols
coh_end   = t1.shape[1] - 3
epi_end   = t1.shape[1]

t1_nli  = t1[:, :nli_end]
t1_emb  = t1[:, nli_end:emb_end]
t1_coh  = t1[:, emb_end:coh_end]
t1_epi  = t1[:, coh_end:]

print(f'\nTier 1 sub-blocks:')
print(f'  NLI       : {t1_nli.shape}')
print(f'  Embedding : {t1_emb.shape}')
print(f'  Coherence : {t1_coh.shape}')
print(f'  Epistemic : {t1_epi.shape}')

In [ ]:
# ── Cell 3: Full Ablation Table ───────────────────────────────────────────────
# Evaluates all tier combinations with LightGBM (5-fold CV).

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def eval_config(X_cfg, y, n_splits=5, random_state=42):
    """5-fold CV with LightGBM. Returns mean metrics dict."""
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    accs, f1s, precs, recs = [], [], [], []
    for train_idx, val_idx in cv.split(X_cfg, y):
        clf = lgb.LGBMClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, reg_alpha=0.1, reg_lambda=0.1,
            random_state=random_state, n_jobs=-1, verbose=-1,
        )
        clf.fit(X_cfg[train_idx], y[train_idx])
        y_pred = clf.predict(X_cfg[val_idx])
        accs.append(accuracy_score(y[val_idx], y_pred))
        f1s.append(f1_score(y[val_idx], y_pred))
        precs.append(precision_score(y[val_idx], y_pred))
        recs.append(recall_score(y[val_idx], y_pred))
    return {
        'accuracy':  np.mean(accs),
        'f1':        np.mean(f1s),
        'precision': np.mean(precs),
        'recall':    np.mean(recs),
    }

ablation_configs = [
    ('Random baseline',       None),                             # ~50%
    ('T1-NLI only',           t1_nli),
    ('T1-Embedding only',     t1_emb),
    ('T1-Coherence only',     t1_coh),
    ('T1-Epistemic only',     t1_epi),
    ('Tier 1 only',           t1),
    ('Tier 2 only',           t2),
    ('Tier 3 only',           t3),
    ('T1 + T2',               np.hstack([t1, t2])),
    ('T1 + T3',               np.hstack([t1, t3])),
    ('T2 + T3',               np.hstack([t2, t3])),
    ('T1 + T2 + T3 (Full)',   np.hstack([t1, t2, t3])),
]

rows = []
for name, X_cfg in ablation_configs:
    if X_cfg is None:
        # Random baseline: predict majority class always
        majority = int(np.bincount(y).argmax())
        preds = np.full(len(y), majority)
        row = {
            'config':    name,
            'accuracy':  accuracy_score(y, preds),
            'f1':        f1_score(y, preds, zero_division=0),
            'precision': precision_score(y, preds, zero_division=0),
            'recall':    recall_score(y, preds, zero_division=0),
        }
    else:
        metrics = eval_config(X_cfg, y)
        row = {'config': name, **metrics}
    rows.append(row)
    print(f"{name:30s}  acc={row['accuracy']:.4f}  f1={row['f1']:.4f}")

df_ablation = pd.DataFrame(rows)
df_ablation = df_ablation.sort_values('accuracy', ascending=False).reset_index(drop=True)

df_ablation.to_csv(f'{PROJECT_DIR}/results/ablation_results.csv', index=False)
print('\nFull ablation table saved.')

In [ ]:
# ── Cell 4: LightGBM vs. Logistic Regression Comparison ──────────────────────

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

X_full = np.hstack([t1, t2, t3])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1', 'precision', 'recall']

# LightGBM
lgb_clf = lgb.LGBMClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, reg_alpha=0.1, reg_lambda=0.1,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgb_cv = cross_validate(lgb_clf, X_full, y, cv=cv, scoring=scoring)

# Logistic Regression (scaled)
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, C=1.0, random_state=42, n_jobs=-1))
])
lr_cv = cross_validate(lr_pipe, X_full, y, cv=cv, scoring=scoring)

rows = []
for model_name, cv_results in [('LightGBM', lgb_cv), ('LogisticRegression', lr_cv)]:
    rows.append({
        'model':     model_name,
        'accuracy':  cv_results['test_accuracy'].mean(),
        'f1':        cv_results['test_f1'].mean(),
        'precision': cv_results['test_precision'].mean(),
        'recall':    cv_results['test_recall'].mean(),
    })

df_compare = pd.DataFrame(rows)
print('Model Comparison (Full feature set, 5-fold CV):')
print(df_compare.round(4).to_string(index=False))
print()

lgb_acc = df_compare.loc[df_compare['model'] == 'LightGBM', 'accuracy'].values[0]
lr_acc  = df_compare.loc[df_compare['model'] == 'LogisticRegression', 'accuracy'].values[0]
delta   = lgb_acc - lr_acc

if abs(delta) < 0.02:
    print('LGB ≈ LR: feature interactions are largely linear.')
else:
    print(f'LGB >> LR by {delta:.4f}: genuine non-linear interactions present.')

df_compare.to_csv(f'{PROJECT_DIR}/results/lgb_vs_lr_comparison.csv', index=False)
print('Comparison saved.')

In [ ]:
# ── Cell 5: Confusion Matrices for Best Config ────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

X_full = np.hstack([t1, t2, t3])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(y))
for train_idx, val_idx in cv.split(X_full, y):
    clf = lgb.LGBMClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, reg_alpha=0.1, reg_lambda=0.1,
        random_state=42, n_jobs=-1, verbose=-1,
    )
    clf.fit(X_full[train_idx], y[train_idx])
    oof_preds[val_idx] = clf.predict(X_full[val_idx])

cm = confusion_matrix(y, oof_preds)
cm_norm = confusion_matrix(y, oof_preds, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

disp1 = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['False (misinformation)', 'True (original)']
)
disp1.plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix (Counts)')

disp2 = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=['False (misinformation)', 'True (original)']
)
disp2.plot(ax=axes[1], colorbar=False)
axes[1].set_title('Confusion Matrix (Normalized)')

plt.suptitle('Full System (T1+T2+T3) — 5-Fold OOF', fontsize=13)
plt.tight_layout()
cm_path = f'{PROJECT_DIR}/results/confusion_matrix_best.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Confusion matrix saved: {cm_path}')

In [ ]:
# ── Cell 6: Print Final Numbers for Paper ────────────────────────────────────

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report
)

print('=' * 65)
print('  FINAL RESULTS — Full System (T1 + T2 + T3), 5-Fold OOF CV')
print('=' * 65)

acc  = accuracy_score(y, oof_preds)
prec = precision_score(y, oof_preds)
rec  = recall_score(y, oof_preds)
f1   = f1_score(y, oof_preds)

print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1        : {f1:.4f}')
print()
print('Per-class report:')
print(classification_report(
    y, oof_preds,
    target_names=['False (misinformation)', 'True (original)']
))

# Ablation summary for Table 3 in paper
print('=' * 65)
print('  ABLATION SUMMARY (for Table 3 in paper)')
print('=' * 65)
df_ablation = pd.read_csv(f'{PROJECT_DIR}/results/ablation_results.csv')
print(df_ablation[['config', 'accuracy', 'f1']].to_string(index=False))

print()
print('=' * 65)
print('  LGB vs LR COMPARISON (for Section 5.2 in paper)')
print('=' * 65)
df_cmp = pd.read_csv(f'{PROJECT_DIR}/results/lgb_vs_lr_comparison.csv')
print(df_cmp.to_string(index=False))

print()
print('All result files saved to:', f'{PROJECT_DIR}/results/')